## Read and Merge FGCM run data

- author : Sylvie Dagoret-Campagne
- creation date : 2026-02-06
- last update : 2026-07-03 : adapted to the new DP2 collection (`LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1..4`),
  `data_fgcm/` layout and `expMidptMJD` / `physical_filter` schema (see `03-ReadFGCMAtmosphericParameters.ipynb`).
  The file discovery is now automatic (glob over `data_fgcm/`) instead of hard-coded filenames from the
  obsolete `DM-53545` / `DM-53697` runs, so the merge step scales to however many `stageN` extractions
  are actually present on disk.

### Links
- doc lsst-pipelines : https://pipelines.lsst.io/v/v23_0_0/modules/lsst.fgcmcal/
- github : https://github.com/lsst/fgcmcal
- runs : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs
- plot-Navigator: https://usdf-rsp.slac.stanford.edu/plot-navigator/collection/dp2_prep/LSSTCam%2Fruns%2FDRP%2F20250417_20250723%2Fd_2025_11_21%2FDM-53374

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
from astropy.table import Table, vstack
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

## Configuration

In [ ]:
plt.rcParams["figure.figsize"] = (16, 6)
plt.rcParams["axes.labelsize"] = "xx-large"
plt.rcParams["axes.titlesize"] = "xx-large"
plt.rcParams["xtick.labelsize"] = "xx-large"
plt.rcParams["ytick.labelsize"] = "xx-large"
plt.rcParams["legend.fontsize"] = "xx-large"

In [ ]:
# Rubin-LSST / Cerro Pachon
lsst = EarthLocation(lat=-30.2417 * u.deg, lon=-70.7366 * u.deg, height=2663 * u.m)

In [ ]:
# where the figures and the output data are stored (renamed 04 -> 05 to match this notebook's number)
pathfigs = "figs_FGCM05_ReadMergeAtmParams"
pathdataout = "data_FGCM05_ReadMergeAtmParams"
prefix = "fgcm05"
for _p in (pathfigs, pathdataout):
    if not os.path.exists(_p):
        os.makedirs(_p)
figtype = ".png"

In [ ]:
# ----- FGCM dataset location (same convention as 03-ReadFGCMAtmosphericParameters.ipynb) -----
REPO_URI = "dp2_prep"
# collection = "LSSTCam/runs/DRP/20250417_20250723/d_2025_11_21/DM-53374"        # 2025-12-05, obsolete
# collection = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"          # 2025-12-12, obsolete
# collection = "LSSTCam/runs/DRP/20250515-20251214/v30_0_0_rc2/DM-53697"        # 2026-02-06, obsolete
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

DATA_DIR = "data_fgcm"
suptitle = f"repo {REPO_URI}, coll = {collection[0]} ..."

## Tools

In [ ]:
def get_table_mjd(t):
    """
    Return an array of MJD values for an FGCM visit table, robust to schema variants:
    prefer the precomputed 'expMidptMJD' column, otherwise derive it from 'expMidpt'
    (isot strings, possibly stored as bytes in FITS).
    """
    if "expMidptMJD" in t.colnames:
        return np.asarray(t["expMidptMJD"], dtype=float)
    raw = t["expMidpt"]
    raw_str = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in raw])
    return Time(raw_str, format="isot", scale="utc").mjd

In [ ]:
# Palette par filtre
default_filter_colors = {
    "u_24": "tab:blue",
    "g_6": "tab:green",
    "r_57": "tab:red",
    "i_39": "tab:orange",
    "z_20": "tab:gray",
    "y_10": "black",
}


def plot_atm_parameter(t_join, param="pwv", filter_colors=None):
    """
    Trace un parametre atmospherique par date, par filtre,
    avec bandes grises = nuit astronomique au site Rubin-LSST.

    Parametres
    ----------
    t_join : astropy.Table
        Table jointe avec colonnes 'physical_filter', 'expMidptMJD' (ou 'expMidpt') et le parametre choisi
    param : str
        Nom du parametre a tracer ('pwv', 'o3', 'tau', 'alpha', ...)
    filter_colors : dict, optional
        Dictionnaire {filter_name: couleur}, sinon palette par defaut
    """
    if filter_colors is None:
        filter_colors = default_filter_colors

    mjd = get_table_mjd(t_join)
    filters = np.asarray(t_join["physical_filter"])
    values = np.asarray(t_join[param], dtype=float)

    mask_valid = np.isfinite(values) & np.isfinite(mjd)
    dates_utc = Time(mjd, format="mjd").to_datetime()

    filter_order = list(filter_colors.keys())

    plt.figure(figsize=(18, 8))

    # Scatter par filtre
    for f in filter_order:
        m = (filters == f) & mask_valid
        if np.sum(m) > 0:
            plt.scatter(dates_utc[m], values[m], s=12, alpha=0.6, color=filter_colors[f], label=f)

    # Fonctions auxiliaires
    def night_astronomical_utc(day_mjd, location):
        t_start = Time(day_mjd, format="mjd")
        t_grid = t_start + np.arange(0, 1.5, 5 / 1440)  # 1.5 jour pour capturer la nuit complete
        altaz = AltAz(obstime=t_grid, location=location)
        sun_alt = get_sun(t_grid).transform_to(altaz).alt
        mask_night = sun_alt < -18 * u.deg
        night_times = t_grid[mask_night]
        if len(night_times) == 0:
            return None, None
        return night_times[0], night_times[-1]

    # Bandes grises = nuit astronomique
    start_day = int(np.floor(mjd[mask_valid].min()))
    end_day = int(np.ceil(mjd[mask_valid].max()))
    all_days = np.arange(start_day, end_day, 1)

    for day_mjd in all_days:
        start_night, end_night = night_astronomical_utc(day_mjd, lsst)
        if start_night is not None:
            plt.axvspan(start_night.datetime, end_night.datetime, color="gray", alpha=0.05)

    # Format axe X
    ax = plt.gca()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d:%H"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=7, maxticks=15))
    ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

    plt.xticks(rotation=45)

    plt.xlabel("Date (UTC)")
    plt.ylabel(f"{param.upper()}")
    plt.title(f"{param.upper()} vs Date (colored by filter)\nGray = astronomical night LSST")
    plt.legend(title="Filter", markerscale=1.5)
    plt.grid(True, alpha=0.3)
    plt.suptitle(suptitle)
    plt.tight_layout()
    figname = f"{pathfigs}/{prefix}_{param}" + figtype
    plt.savefig(figname)
    plt.show()

## Start

## Read files

In [ ]:
# List what is actually available in data_fgcm/ (fits extractions only, one per DRP stage)
!ls -la data_fgcm

In [ ]:
# Choix du format : "fits" ou "ecsv"
format_save = "fits"  # ou "ecsv"

# Discover all FGCM stage extractions automatically instead of hard-coding filenames:
# this is what makes the merge step below scale as more DM-53881/stageN files get produced.
pattern = os.path.join(
    DATA_DIR, f"fgcm_rdp2_prep_cLSSTCam_runs_DRP_DP2_v30_0_0_DM-53881_stage*.{format_save}"
)
all_filenames = sorted(glob.glob(pattern))
print(f"Found {len(all_filenames)} file(s):")
for fn in all_filenames:
    print(" -", fn)

In [ ]:
if len(all_filenames) == 0:
    raise FileNotFoundError(
        f"No FGCM extraction found matching {pattern!r}. "
        "Check that data_fgcm/ contains the expected fgcm_rdp2_prep_..._stageN_*.fits files."
    )

In [ ]:
filename_out = f"{pathdataout}/fgcm_rdp2_prep_cLSSTCam_runs_DRP_DP2_v30_0_0_DM-53881_merged_stages.csv"

### Read all datafiles

In [ ]:
t_join_loaded = [Table.read(filename) for filename in all_filenames]

### Concatenate all tables

In [ ]:
# Add a column to remember which DRP stage produced each row.
# The stage tag is parsed directly from the filename (stage1, stage2, ...) rather than
# from the enumeration order, so it stays correct however many/whichever files are found.
stage_re = re.compile(r"stage(\d+)")

t_join = []
for filename, t in zip(all_filenames, t_join_loaded):
    m = stage_re.search(os.path.basename(filename))
    run_tag = f"stage{m.group(1)}" if m else "unknown"
    t["run"] = np.full(len(t), run_tag)
    t_join.append(t)

In [ ]:
t_join = vstack(t_join)
t_join.sort("visit")

### Show the Schema of the table

In [ ]:
for col in t_join.columns.values():
    print(col.name, col.dtype, col.unit, col.description)

In [ ]:
t_join

In [ ]:
print(t_join.colnames)
print(t_join[:2])

In [ ]:
[name for name in t_join.colnames if "pmb" in name]

### Check the physical filters

In [ ]:
unique_filters = np.unique(t_join["physical_filter"])
print(unique_filters)

In [ ]:
# how many visits come from each DRP stage
unique_runs, counts = np.unique(t_join["run"], return_counts=True)
dict(zip(unique_runs, counts))

## Save in output file

In [ ]:
names = [name for name in t_join.colnames if len(t_join[name].shape) <= 1]
df = t_join[names].to_pandas()
mjd = get_table_mjd(t_join)
df["expMidptMJD"] = mjd
df["Time"] = Time(mjd, format="mjd").to_datetime()

In [ ]:
df.to_csv(filename_out, index=False)
print(f"Wrote {len(df)} rows to {filename_out}")

In [ ]:
df

## Plots

### Plot PWV

In [ ]:
plot_atm_parameter(t_join, param="pwv")

### Plot Ozone

In [ ]:
plot_atm_parameter(t_join, param="o3")

### Plot tau

In [ ]:
plot_atm_parameter(t_join, param="tau")

### Plot alpha

In [ ]:
plot_atm_parameter(t_join, param="alpha")